In [1]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

print(train.shape)
print(test.shape)

(440679, 3)
(48965, 3)


In [2]:
print(train.head())
print(train.columns.tolist())

                                                text     label         dataset
0  ürünü hepsiburadadan alalı 3 hafta oldu. orjin...  Positive  urun_yorumlari
1  ürünlerden çok memnunum, kesinlikle herkese ta...  Positive  urun_yorumlari
2      hızlı kargo, temiz alışveriş.teşekkür ederim.  Positive  urun_yorumlari
3               Çünkü aranan tapınak bu bölgededir .      Notr            wiki
4  bu telefonu başlıca alma nedenlerim ise elimde...  Positive  urun_yorumlari
['text', 'label', 'dataset']


In [3]:
print(train["label"].value_counts())

label
Positive    235949
Notr        153825
Negative     50905
Name: count, dtype: int64


In [4]:
print(train["dataset"].value_counts())

dataset
urun_yorumlari      210693
wiki                153364
HUMIR                58575
tweet-pn              9959
magaza_yorumlari      7627
random                 461
Name: count, dtype: int64


In [5]:
import yfinance as yf

# BIST hisselerinde .IS eki var
thyao = yf.download("THYAO.IS", start="2024-01-01", end="2024-12-31")
print(thyao.head())
print(thyao.shape)

[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Ticker        THYAO.IS    THYAO.IS    THYAO.IS    THYAO.IS  THYAO.IS
Date                                                                
2024-01-02  230.935410  232.303056  228.102451  228.102451  25232547
2024-01-03  224.781052  230.544662  224.390289  230.446968  19679997
2024-01-04  227.907089  228.883973  224.390300  225.367184  20462665
2024-01-05  233.670685  233.670685  227.809380  228.395516  31168225
2024-01-08  236.992111  238.457437  232.205370  233.573017  26271851
(248, 5)


In [6]:
# Fiyat değişimini hesapla
thyao["return_2d"] = thyao["Close"].pct_change(periods=2).shift(-2)

print(thyao[["Close", "return_2d"]].head(10))

Price            Close return_2d
Ticker        THYAO.IS          
Date                            
2024-01-02  230.935410 -0.013113
2024-01-03  224.781052  0.039548
2024-01-04  227.907089  0.039863
2024-01-05  233.670685  0.015468
2024-01-08  236.992111  0.030091
2024-01-09  237.285156  0.015644
2024-01-10  244.123352  0.004402
2024-01-11  240.997330  0.026550
2024-01-12  245.197922 -0.004781
2024-01-15  247.395920 -0.002962


In [7]:
def etiketle(getiri):
    if getiri > 0.02:       # +%2'den fazla
        return "bullish"
    elif getiri < -0.02:    # -%2'den fazla
        return "bearish"
    else:
        return "neutral"

thyao["etiket"] = thyao["return_2d"].apply(etiketle)

print(thyao["etiket"].value_counts())

etiket
neutral    137
bullish     64
bearish     47
Name: count, dtype: int64


In [8]:
import requests
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("NEWS_API_KEY")

url = "https://newsapi.org/v2/everything"
params = {
    "q": "borsa hisse yatırım",
    "language": "tr",
    "pageSize": 10,
    "apiKey": api_key
}

response = requests.get(url, params=params)
print(response.status_code)
print(response.json().keys())

200
dict_keys(['status', 'totalResults', 'articles'])


In [9]:
haberler = response.json()["articles"]
print(f"Toplam haber: {len(haberler)}")
print("---")
print(haberler[0]["title"])
print(haberler[0]["description"])
print(haberler[0]["publishedAt"])

Toplam haber: 10
---
Yatırım araçlarının haftalık performansı ne oldu?
Borsa İstanbul'da işlem gören hisse senetleri haftalık bazda ortalama yüzde 0,35, altının gram fiyatı yüzde 7,45 değer kaybederken, dolar/TL yüzde 0,30 ve euro/TL yüzde 0,16 değer kazandı.
2026-03-19T11:53:00Z


In [10]:
import pandas as pd

# Haberleri düzenli bir listeye çevir
temiz_haberler = []
for haber in haberler:
    temiz_haberler.append({
        "tarih": haber["publishedAt"],
        "baslik": haber["title"],
        "aciklama": haber["description"],
        "kaynak": haber["source"]["name"]
    })

df = pd.DataFrame(temiz_haberler)
print(df)

                  tarih                                             baslik  \
0  2026-03-19T11:53:00Z  Yatırım araçlarının haftalık performansı ne oldu?   
1  2026-03-19T11:17:00Z  Hangi yatırım aracı ne kadar kazandırdı? Detay...   
2  2026-02-28T10:52:00Z  10 yıl önce 100 bin lira yatırım yapan bir kiş...   
3  2026-03-14T07:19:00Z  Yatırımcıların yüzü güldü: Haftanın en çok kaz...   
4  2026-03-06T13:59:00Z  Borsada temettü yağmuru: Dev şirketlerden naki...   
5  2026-03-19T13:29:07Z  Borsa İstanbul’da 'Tera' kararı: Kriterler tut...   
6  2026-03-28T08:10:30Z  'Türkiye 60 ton altın sattı' denildi... Savaşı...   
7  2026-03-19T10:52:08Z  Haftanın en çok kazandıran yatırım araçları be...   
8  2026-03-13T16:10:15Z  Euro bu hafta yatırım araçları arasında tek de...   
9  2026-02-27T17:31:41Z  Yatırım araçlarının haftalık performansında al...   

                                            aciklama                kaynak  
0  Borsa İstanbul'da işlem gören hisse senetleri ...            

In [11]:
import os

os.makedirs("../data/raw", exist_ok=True)
df.to_csv("../data/raw/haberler.csv", index=False, encoding="utf-8-sig")
print("Kaydedildi.")

Kaydedildi.


In [13]:
import time

arama_terimleri = [
    "borsa İstanbul",
    "hisse senedi",
    "Türkiye ekonomi",
    "faiz enflasyon",
    "şirket kâr",
    "dolar TL kur",
    "yatırım fon",
    "temettü"
]

tum_haberler = []

for terim in arama_terimleri:
    params = {
        "q": terim,
        "language": "tr",
        "pageSize": 100,
        "apiKey": api_key
    }

    response = requests.get("https://newsapi.org/v2/everything", params=params)

    if response.status_code == 200:
        haberler = response.json()["articles"]
        for haber in haberler:
            tum_haberler.append({
                "tarih": haber["publishedAt"],
                "baslik": haber["title"],
                "aciklama": haber["description"],
                "kaynak": haber["source"]["name"]
            })
        print(f"'{terim}' → {len(haberler)} haber")
    else:
        print(f"Hata: {response.status_code}")

    time.sleep(1)  # API'ye çok hızlı istek atmayalım

print(f"\nToplam: {len(tum_haberler)} haber")

'borsa İstanbul' → 100 haber
'hisse senedi' → 90 haber
'Türkiye ekonomi' → 100 haber
'faiz enflasyon' → 100 haber
'şirket kâr' → 34 haber
'dolar TL kur' → 36 haber
'yatırım fon' → 21 haber
'temettü' → 27 haber

Toplam: 508 haber


In [14]:
df = pd.DataFrame(tum_haberler)

# Tekrar edenleri at — aynı başlık birden fazla kez gelmiş olabilir
onceki = len(df)
df = df.drop_duplicates(subset="baslik")
sonraki = len(df)

print(f"Tekrar eden: {onceki - sonraki} haber atıldı")
print(f"Kalan: {sonraki} haber")

# Kaydet
df.to_csv("../data/raw/haberler.csv", index=False, encoding="utf-8-sig")
print("Kaydedildi.")

Tekrar eden: 48 haber atıldı
Kalan: 460 haber
Kaydedildi.
